# Сравнение стратегий формирования эпизодов: Германия — Шотландия

В этом ноутбуке строится такой же презентационный график, как в общем эксперименте, но только для матча **Germany — Scotland** (`match_id=3930158`).

Используются уже сохранённые payloads для LLM из папки `outputs/chain_grouping_llm_model_experiments`: один JSON-файл на стратегию группировки. Для графика оставлены стратегии:

- `event_only` — каждое событие отдельно;
- `related_only` — цепочки по `related_events` без обрезки;
- `related_cap` — `related_events` + лимиты длины/времени;
- `related_stop_cap` — `related_cap` + стоп-события;
- `time_window` — группировка по временным окнам.

`possession_cap` здесь не показываем, чтобы график соответствовал финальной версии ВКР.

In [ ]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
OUT_DIR = ROOT / 'outputs' / 'chain_grouping_llm_model_experiments'
FIG_DIR = OUT_DIR / 'figures'
TAB_DIR = OUT_DIR / 'tables'
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

MATCH_ID = '3930158'  # Germany — Scotland

STRATEGY_ORDER = [
    'event_only',
    'related_only',
    'related_cap',
    'related_stop_cap',
    'time_window',
]

label_ru = {
    'event_only': 'Событие\nкак есть',
    'related_only': 'Related\nбез обрезки',
    'related_cap': 'Related\n+ лимиты',
    'related_stop_cap': 'Related\n+ стоп-события',
    'time_window': 'Окно\nпо времени',
}

PAYLOAD_FILES = {
    st: OUT_DIR / f'llm_payloads_speak_set_{st}_summary.json'
    for st in STRATEGY_ORDER
}

PAYLOAD_FILES

## Загрузка payloads

Фильтруем каждый файл по `match_id=3930158`. В этих JSON уже лежат payloads, которые отбирались для LLM (`brief`/`must`), поэтому правая панель графика показывает именно **сколько payloads ушло бы в LLM для этого матча**.

In [ ]:
def load_strategy_payloads(strategy: str, path: Path, match_id: str = MATCH_ID):
    if not path.exists():
        raise FileNotFoundError(path)
    data = json.loads(path.read_text(encoding='utf-8'))
    return [p for p in data if str(p.get('match_id')) == str(match_id)]

payloads_by_strategy = {
    st: load_strategy_payloads(st, path)
    for st, path in PAYLOAD_FILES.items()
}

for st, payloads in payloads_by_strategy.items():
    print(f'{st:18s}: {len(payloads)} payloads')

## Расчёт статистик

Для каждой стратегии считаем:

- `chains_n` — число payloads для LLM;
- `events_median` — типичная длина payload в событиях;
- `events_p90` — 90-й процентиль длины, то есть хвост длинных цепочек;
- `duration_*` — длительность payload по timestamp первого и последнего события;
- `json_chars_*` — примерный размер JSON-входа.

In [ ]:
def to_seconds(ts):
    if not ts or not isinstance(ts, str):
        return None
    m = re.match(r'^(\d+):(\d+):(\d+)(?:\.(\d+))?$', ts.strip())
    if not m:
        return None
    hh, mm, ss, ms = m.groups()
    base = int(hh) * 3600 + int(mm) * 60 + int(ss)
    frac = float(f'0.{ms}') if ms else 0.0
    return base + frac


def payload_duration_sec(p):
    events = p.get('events') or []
    if not events:
        return 0.0
    t0 = to_seconds(((events[0].get('event_json') or {}).get('timestamp')))
    t1 = to_seconds(((events[-1].get('event_json') or {}).get('timestamp')))
    return max(0.0, (t1 - t0)) if (t0 is not None and t1 is not None) else 0.0


def strategy_stats(payloads_by_strategy):
    rows = []
    for st, payloads in payloads_by_strategy.items():
        lens = np.array([len(p.get('events', [])) for p in payloads], dtype=float)
        durs = np.array([payload_duration_sec(p) for p in payloads], dtype=float)
        sizes = np.array([len(json.dumps(p, ensure_ascii=False)) for p in payloads], dtype=float)
        rows.append({
            'strategy': st,
            'chains_n': len(payloads),
            'events_mean': lens.mean() if len(lens) else 0,
            'events_median': np.median(lens) if len(lens) else 0,
            'events_p90': np.percentile(lens, 90) if len(lens) else 0,
            'duration_mean': durs.mean() if len(durs) else 0,
            'duration_median': np.median(durs) if len(durs) else 0,
            'duration_p90': np.percentile(durs, 90) if len(durs) else 0,
            'json_chars_mean': sizes.mean() if len(sizes) else 0,
            'json_chars_p90': np.percentile(sizes, 90) if len(sizes) else 0,
        })
    return pd.DataFrame(rows).sort_values('strategy')


stats_df = strategy_stats(payloads_by_strategy)
stats_df['strategy'] = pd.Categorical(stats_df['strategy'], categories=STRATEGY_ORDER, ordered=True)
stats_df = stats_df.sort_values('strategy').reset_index(drop=True)
stats_df['strategy_ru'] = stats_df['strategy'].astype(str).map(label_ru)

out_table = TAB_DIR / 'germany_scotland_grouping_strategy_stats.csv'
stats_df.to_csv(out_table, index=False)
print('saved:', out_table)
display(stats_df)

## Презентационный график

Левая панель показывает компактность payloads: медиану и 90-й процентиль длины. Правая панель показывает «цену» стратегии — сколько payloads будет отправлено в LLM для одного матча Германия — Шотландия.

In [ ]:
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'axes.titlesize': 15,
    'xtick.labelsize': 10,
    'ytick.labelsize': 11,
    'mathtext.fontset': 'dejavusans',
})

hse_blue = '#005BBB'
bar_blue = '#0B63B6'
light_blue = '#D7ECFF'
accent = '#E84A5F'
grid = '#D9E2EC'
text = '#1F2937'

plot_df = stats_df.copy()

fig, axes = plt.subplots(
    1, 2,
    figsize=(15.8, 6.4),
    gridspec_kw={'width_ratios': [1.45, 1.0]},
    constrained_layout=True,
)
fig.patch.set_facecolor('white')
fig.suptitle(
    'Сравнение стратегий формирования эпизодов\nGermany — Scotland',
    x=0.5, y=1.08, ha='center', va='bottom',
    fontsize=20, fontweight='bold', color=text,
)

ax = axes[0]
x = np.arange(len(plot_df))
median = plot_df['events_median'].to_numpy(dtype=float)
p90 = plot_df['events_p90'].to_numpy(dtype=float)
ymax = max(float(np.nanmax(p90)) if len(p90) else 1.0, 1.0)

ax.bar(
    x,
    median,
    width=0.62,
    color=bar_blue,
    edgecolor='#083B75',
    linewidth=1.1,
    label=r'Медиана $P_{50}$',
    zorder=3,
)

for xi, med, q90 in zip(x, median, p90):
    ax.plot([xi, xi], [med, q90], color=accent, linewidth=2.3, zorder=4)
    ax.plot([xi - 0.10, xi + 0.10], [q90, q90], color=accent, linewidth=2.3, zorder=4)
    ax.scatter([xi], [q90], s=58, color=accent, edgecolor='white', linewidth=1.1, zorder=5)

for xi, med, q90 in zip(x, median, p90):
    if med > 0:
        ax.text(
            xi,
            max(med * 0.52, 0.20),
            rf'$P_{{50}}={med:.0f}$',
            ha='center', va='center',
            color='white', fontsize=10, fontweight='bold',
            zorder=6,
        )
    else:
        ax.text(xi, 0.08, r'$P_{50}=0$', ha='center', va='bottom', color=text, fontsize=10, zorder=6)

    ax.text(
        xi,
        q90 + ymax * 0.045,
        rf'$P_{{90}}={q90:.0f}$',
        ha='center', va='bottom',
        color=accent, fontsize=10, fontweight='bold',
        zorder=6,
    )

median_proxy = plt.Rectangle((0, 0), 1, 1, fc=bar_blue, ec='#083B75', label=r'Медиана $P_{50}$')
p90_proxy = plt.Line2D([0], [0], color=accent, marker='o', markersize=7, linewidth=2.3, label=r'Квантиль $P_{90}$')
ax.legend(handles=[median_proxy, p90_proxy], frameon=False, loc='upper right')

ax.set_title('Длина эпизода по стратегиям группировки', loc='left', color=text, pad=16)
ax.set_ylabel('Событий в одном payload')
ax.set_xticks(x)
ax.set_xticklabels(plot_df['strategy_ru'])
ax.set_ylim(0, ymax * 1.24)
ax.grid(axis='y', color=grid, linewidth=1.0, alpha=0.8, zorder=0)
ax.spines[['top', 'right']].set_visible(False)
ax.spines[['left', 'bottom']].set_color('#B8C2CC')

ax2 = axes[1]
chains = plot_df['chains_n'].to_numpy(dtype=float)
bars2 = ax2.barh(
    x,
    chains,
    height=0.58,
    color=light_blue,
    edgecolor=hse_blue,
    linewidth=1.2,
    zorder=3,
)
for b, val in zip(bars2, chains):
    ax2.text(
        val + max(chains.max(), 1) * 0.015,
        b.get_y() + b.get_height() / 2,
        f'{int(val):,}'.replace(',', ' '),
        va='center', ha='left',
        color=text,
        fontsize=10,
        fontweight='bold',
    )

ax2.set_title('Сколько payloads уйдёт в LLM', loc='left', color=text, pad=16)
ax2.set_xlabel('Количество payloads')
ax2.set_yticks(x)
ax2.set_yticklabels(plot_df['strategy_ru'])
ax2.invert_yaxis()
ax2.grid(axis='x', color=grid, linewidth=1.0, alpha=0.8, zorder=0)
ax2.spines[['top', 'right']].set_visible(False)
ax2.spines[['left', 'bottom']].set_color('#B8C2CC')

out_plot = FIG_DIR / 'germany_scotland_grouping_strategy_compactness.png'
fig.savefig(out_plot, dpi=220, bbox_inches='tight', facecolor='white')
plt.show()
print('saved:', out_plot)